In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/home/duarte/Desktop/Tese/Mapping_Tese/mapping_tese")

CNN_CSV = PROJECT_ROOT / "notebooks/CNN/results/cnn_loro_participant_results.csv"
ANN_CSV = PROJECT_ROOT / ("notebooks/ANN/results/4_Classes_ANN_MultiSubject/all_subjects_results.csv")
OUT_DIR = PROJECT_ROOT / "images/CNN_vs_ANN"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CNN_RESULTS_DIR = PROJECT_ROOT / "notebooks/CNN/results"
ANN_RESULTS_DIR = PROJECT_ROOT / ("notebooks/ANN/results/4_Classes_ANN_MultiSubject")

In [ ]:
import pandas as pd
import numpy as np

cnn_df = pd.read_csv(CNN_CSV)[["subject", "mean_balanced_accuracy"]].rename(columns={"mean_balanced_accuracy": "cnn_bal_acc"})
ann_df = pd.read_csv(ANN_CSV)[["subject", "mean_balanced_accuracy"]].rename(columns={"mean_balanced_accuracy": "ann_bal_acc"})

merged = cnn_df.merge(
    ann_df,
    on="subject",
    how="inner",
    validate="one_to_one",
)

merged["cnn_bal_acc_pct"] = merged["cnn_bal_acc"] * 100.0
merged["ann_bal_acc_pct"] = merged["ann_bal_acc"] * 100.0
merged["delta_ann_minus_cnn"] = (merged["ann_bal_acc"] - merged["cnn_bal_acc"])
merged["delta_ann_minus_cnn_pct"] = (merged["delta_ann_minus_cnn"] * 100.0)

eps = 1e-12
merged["winner"] = np.select([merged["delta_ann_minus_cnn"] > eps, merged["delta_ann_minus_cnn"] < -eps,],["ANN", "CNN"],default="Tie",)

print("Subjects in merged table:", len(merged))
display(merged.sort_values("subject").reset_index(drop=True))

In [ ]:
def load_cnn_folds(csv_path):
    df = pd.read_csv(csv_path)

    required = {"subject", "test_run", "balanced_accuracy"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")

    out = df[["subject", "test_run", "balanced_accuracy"]].copy()
    out = out.rename(columns={"test_run": "fold"})
    out["model"] = "CNN"
    return out

In [ ]:
def load_ann_folds(base_dir):
    rows = []

    for csv_path in sorted(base_dir.glob("sub-*/fold_balanced_accuracy.csv")):
        df = pd.read_csv(csv_path)

        required = {"subject", "fold", "balanced_accuracy"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")

        out = df[["subject", "fold", "balanced_accuracy"]].copy()
        out["model"] = "ANN"
        rows.append(out)

    if not rows:
        raise FileNotFoundError("No ANN fold_balanced_accuracy.csv files found.")

    return pd.concat(rows, ignore_index=True)

In [ ]:
cnn_fold_df = load_cnn_folds(CNN_RESULTS_DIR / "cnn_loro_fold_results.csv")
ann_fold_df = load_ann_folds(ANN_RESULTS_DIR)

for fold_df in [cnn_fold_df, ann_fold_df]:
    fold_df["subject"] = fold_df["subject"].astype(str)
    fold_df["fold"] = fold_df["fold"].astype(int)
    fold_df["balanced_accuracy"] = fold_df["balanced_accuracy"].astype(float)

shared_subjects = sorted(set(cnn_fold_df["subject"]) & set(ann_fold_df["subject"]))
fold_plot_df = pd.concat([cnn_fold_df[cnn_fold_df["subject"].isin(shared_subjects)], ann_fold_df[ann_fold_df["subject"].isin(shared_subjects)],],ignore_index=True)
fold_plot_df["balanced_accuracy_pct"] = (fold_plot_df["balanced_accuracy"] * 100.0)

fold_plot_df["model"] = pd.Categorical(fold_plot_df["model"], categories=["CNN", "ANN"], ordered=True,)
fold_plot_df = fold_plot_df.sort_values(["subject", "model", "fold"]).reset_index(drop=True)
fold_plot_df.to_csv(OUT_DIR / "per_fold_balanced_accuracy_long.csv", index=False)

print("Shared subjects:", shared_subjects)
print("Rows:", len(fold_plot_df))
print("Expected rows:", len(shared_subjects) * 4 * 2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg

sns.set_theme(style="whitegrid", context="talk")

check_counts = fold_plot_df.groupby(["subject", "model"])["fold"].nunique().reset_index(name="n_folds")
bad = check_counts[check_counts["n_folds"] != 4]

if not bad.empty:
    print("Warning: some subject/model pairs do not have exactly 4 folds:")
    display(bad)

display(fold_plot_df.head(12))

In [ ]:
palette = {"CNN": "#4C78A8", "ANN": "#F58518"}

fold_plot_df_plot = fold_plot_df.copy()
fold_plot_df_plot["_group"] = "All subjects"

fig, ax = plt.subplots(figsize=(5, 6), dpi=150)

sns.violinplot(data=fold_plot_df_plot, x="_group", y="balanced_accuracy_pct", hue="model", split=True, palette=palette, inner="quart", cut=0, linewidth=1.2, ax=ax)

ax.axhline(25, color="red", linestyle="--", linewidth=1.2, label="Chance (25%)")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title(f"Fold Balanced Accuracy: CNN vs ANN\n({len(shared_subjects)} subjects × 4 folds each)")
ax.legend(title="Model", loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig(OUT_DIR / "violin_fold_balanced_accuracy_cnn_vs_ann.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
per_subject_table = merged[["subject", "cnn_bal_acc_pct", "ann_bal_acc_pct", "delta_ann_minus_cnn_pct", "winner"]].copy()

per_subject_table = per_subject_table.sort_values("delta_ann_minus_cnn_pct", ascending=False).reset_index(drop=True)
per_subject_table_rounded = per_subject_table.copy()
per_subject_table_rounded[["cnn_bal_acc_pct", "ann_bal_acc_pct", "delta_ann_minus_cnn_pct"]] = per_subject_table_rounded[["cnn_bal_acc_pct", "ann_bal_acc_pct", "delta_ann_minus_cnn_pct"]].round(2)

per_subject_table_rounded.to_csv(OUT_DIR / "per_subject_balanced_accuracy_table_rounded.csv", index=False)
display(per_subject_table_rounded)

print("ANN wins:", int((per_subject_table["winner"] == "ANN").sum()))
print("CNN wins:", int((per_subject_table["winner"] == "CNN").sum()))
print("Ties:", int((per_subject_table["winner"] == "Tie").sum()))
print("Mean delta (ANN - CNN), percentage points:", round(per_subject_table["delta_ann_minus_cnn_pct"].mean(), 2))

In [ ]:
x = merged["cnn_bal_acc"].to_numpy(dtype=float)
y = merged["ann_bal_acc"].to_numpy(dtype=float)
d = y - x

normality_df = pg.normality(d)
ttest_df = pg.ttest(y, x, paired=True)
wilcoxon_df = pg.wilcoxon(y, x, alternative="two-sided")

normality_df.to_csv(OUT_DIR / "stats_normality_diff.csv", index=False)
ttest_df.to_csv(OUT_DIR / "stats_paired_ttest.csv", index=False)
wilcoxon_df.to_csv(OUT_DIR / "stats_wilcoxon.csv", index=False)

display(normality_df)
display(ttest_df)
display(wilcoxon_df)

In [ ]:
def pick_value(df, candidates, default=np.nan):
    if df.empty:
        return float(default)
    for c in candidates:
        if c in df.columns:
            try:
                return float(df[c].iloc[0])
            except Exception:
                return float(default)
    return float(default)

In [ ]:
paired_t_p = pick_value(ttest_df, ["p-val", "p_val", "pvalue", "p"])
paired_t_d = pick_value(ttest_df, ["cohen-d", "cohen_d", "cohend", "d"])
wilcoxon_p = pick_value(wilcoxon_df, ["p-val", "p_val", "pvalue", "p"])
summary_df = pd.DataFrame([{"n_subjects": len(merged), "cnn_mean_bal_acc": float(np.mean(x)), "cnn_std_bal_acc": float(np.std(x, ddof=1)), "ann_mean_bal_acc": float(np.mean(y)), "ann_std_bal_acc": float(np.std(y, ddof=1)), "mean_diff_ann_minus_cnn": float(np.mean(d)), "mean_diff_percent_points": float(np.mean(d) * 100.0), "paired_t_p": paired_t_p, "paired_t_cohen_d": paired_t_d, "wilcoxon_p": wilcoxon_p}])
summary_df.to_csv(OUT_DIR / "balanced_accuracy_stats_summary.csv", index=False)

print("Normality test of paired differences:")
display(normality_df)
print("Paired t-test:")
display(ttest_df)
print("Wilcoxon signed-rank:")
display(wilcoxon_df)
print("Compact summary:")
display(summary_df)

In [ ]:
plot_df = merged.sort_values("delta_ann_minus_cnn_pct", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 0.6 * len(plot_df) + 2), dpi=150)
y_pos = np.arange(len(plot_df))

for i, row in plot_df.iterrows():
    ax.plot([row["cnn_bal_acc_pct"], row["ann_bal_acc_pct"]], [i, i], color="gray", alpha=0.7, linewidth=2)

ax.scatter(plot_df["cnn_bal_acc_pct"], y_pos, s=70, color="#4C78A8", label="CNN", zorder=3)
ax.scatter(plot_df["ann_bal_acc_pct"], y_pos, s=70, color="#F58518", label="ANN", zorder=3)

ax.axvline(25, color="red", linestyle="--", linewidth=1.2, label="Chance (25%)")
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["subject"])
ax.set_xlabel("Balanced Accuracy (%)")
ax.set_title("Per-subject Balanced Accuracy: CNN vs ANN")
ax.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.savefig(OUT_DIR / "per_subject_dumbbell_cnn_vs_ann.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
delta_df = merged.sort_values("delta_ann_minus_cnn_pct", ascending=False).reset_index(drop=True)
colors = np.where(delta_df["delta_ann_minus_cnn_pct"] >= 0, "#2E8B57", "#B22222")

fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
bars = ax.bar(delta_df["subject"], delta_df["delta_ann_minus_cnn_pct"], color=colors, alpha=0.9)

ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Delta (ANN - CNN), percentage points")
ax.set_title("Per-subject Change in Balanced Accuracy")
ax.tick_params(axis="x", rotation=45)

for b, val in zip(bars, delta_df["delta_ann_minus_cnn_pct"]):
    ax.text(b.get_x() + b.get_width() / 2, val + (0.4 if val >= 0 else -0.6), f"{val:.1f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "per_subject_delta_ann_minus_cnn.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
lines = []
lines.append("Per-subject CNN vs ANN (Balanced Accuracy)")
lines.append("")

for _, r in per_subject_table_rounded.iterrows():
    lines.append(f"{r['subject']}: CNN={r['cnn_bal_acc_pct']:.2f}%, ANN={r['ann_bal_acc_pct']:.2f}%, Delta={r['delta_ann_minus_cnn_pct']:+.2f} pp, Winner={r['winner']}")

lines.append("")
lines.append(f"ANN wins: {int((per_subject_table['winner'] == 'ANN').sum())}")
lines.append(f"CNN wins: {int((per_subject_table['winner'] == 'CNN').sum())}")
lines.append(f"Ties: {int((per_subject_table['winner'] == 'Tie').sum())}")
lines.append(f"Mean Delta (ANN-CNN): {per_subject_table['delta_ann_minus_cnn_pct'].mean():+.2f} pp")
lines.append(f"Paired t-test p: {summary_df.loc[0, 'paired_t_p']:.6g}")
lines.append(f"Wilcoxon p: {summary_df.loc[0, 'wilcoxon_p']:.6g}")

report_path = OUT_DIR / "per_subject_report.txt"
report_path.write_text("\n".join(lines), encoding="utf-8")

print("Saved:", report_path)
print("")
print("\n".join(lines[:8]))